# Stage 2: Two-Tower Model Deployment

Promotes a trained model from Stage 1 (`03_two_tower_train_log`) to production:
1. Loads the item tower pyfunc from the Stage 1 MLflow run
2. Generates item embeddings → writes to Delta → syncs Vector Search index
3. Logs the user tower **retrieval** pyfunc (embedding + VS) via `fe.log_model`
4. Registers to Unity Catalog and deploys to a serving endpoint with route optimization

The Stage 1 run ID is read from `config.yml` (`run_id_to_deploy`). Feature specs are extracted dynamically from the Stage 1 model artifacts — no hardcoded feature lists.

In [0]:
%pip install databricks-feature-engineering>=0.16.0 databricks-vectorsearch
dbutils.library.restartPython()

In [0]:
import yaml, os, sys, json, pickle, tempfile
import importlib.util
import time
import numpy as np
import torch
import pandas as pd
import mlflow
import mlflow.artifacts
from mlflow.models.resources import DatabricksVectorSearchIndex
from pyspark.sql import functions as F
from databricks.sdk import WorkspaceClient
from databricks.feature_engineering import FeatureEngineeringClient
from databricks.vector_search.client import VectorSearchClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput, ServedEntityInput,
)

from transforms import transform_item_features

In [0]:
fe = FeatureEngineeringClient()
w = WorkspaceClient()

In [0]:

# Load project config
config_path = os.path.join(os.getcwd(), "config.yml")
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

CATALOG = config["catalog"]
SCHEMA = config["schema"]
EMBED_DIM = config["embed_dim"]
VS_ENDPOINT = config["vs_endpoint"]
VS_INDEX_NAME = config["vs_index_name"]
VS_INDEX = f"{CATALOG}.{SCHEMA}.{VS_INDEX_NAME}"
EMBEDDING_TABLE = f"{CATALOG}.{SCHEMA}._two_tower_item_embeddings"
PRODUCT_TRAIN_TABLE = f"{CATALOG}.{SCHEMA}.{config['product_train_table']}"
SESSION_TRAIN_TABLE = f"{CATALOG}.{SCHEMA}.{config['session_train_table']}"
UC_MODEL_NAME = f"{CATALOG}.{SCHEMA}.two_tower_user_retrieval"
ENDPOINT_NAME = "two-tower-recommender"

# Add project root to path for pyfunc imports
_project_root = os.path.normpath(os.path.dirname(config_path))
sys.path.insert(0, _project_root)

transforms_module_path = os.path.join(_project_root, "transforms.py")
retrieval_pyfunc_module = os.path.join(_project_root, "user_retrieval_pyfunc.py")

# MLflow experiment
mlflow.set_experiment(config["mlflow_experiment"])

# Stage 1 run to promote (set in config.yml)
STAGE1_RUN_ID = config["run_id_to_deploy"]
assert STAGE1_RUN_ID, "Set run_id_to_deploy in config.yml before running."
print(f"Promoting Stage 1 run: {STAGE1_RUN_ID}")
print(f"Target endpoint: {ENDPOINT_NAME}")
print(f"UC model: {UC_MODEL_NAME}")

### Load item tower

Load the item tower pyfunc from the Stage 1 MLflow run (transforms + vocab encoding + traced tower). Uses the raw inner pyfunc (without FE wrapper) for batch embedding generation — on serverless, the FE-wrapped `predict()` triggers `score_batch` with a local URI which is not supported. Since we already have all item features in the product train table, no FE lookup is needed.

In [0]:
_raw_model_uri = f"runs:/{STAGE1_RUN_ID}/item_tower_pyfunc/data/feature_store/raw_model"
item_model = mlflow.pyfunc.load_model(_raw_model_uri)
print(f"Item tower raw pyfunc loaded from {_raw_model_uri}")

# Also load the pyfunc config and vocabs for downstream use
artifact_dir = mlflow.artifacts.download_artifacts(run_id=STAGE1_RUN_ID, artifact_path="item_tower_pyfunc")
_raw_model = os.path.join(artifact_dir, "data", "feature_store", "raw_model")
with open(os.path.join(_raw_model, "artifacts", "pyfunc_config.json"), "r") as f:
    pyfunc_config = json.load(f)
with open(os.path.join(_raw_model, "artifacts", "vocabs.pkl"), "rb") as f:
    vocabs = pickle.load(f)

print(f"Config: embed_dim={pyfunc_config['embed_dim']}, VS={pyfunc_config['vs_endpoint']}")
print(f"Vocabs: {len(vocabs['item_vocab']):,} items, {len(vocabs['category_vocab'])} cat_l1")

### Generate item embeddings

Generate item embeddings using the Stage 1 item tower pyfunc. The pyfunc handles transforms + vocab encoding internally.

In [0]:
# Load training data to get unique items (use product train table)
train_df = spark.table(PRODUCT_TRAIN_TABLE).toPandas()

# Deduplicate: one row per product with raw item FV columns
raw_item_cols = [
    "product_id", "item_price", "item_log_price",
    "item_category_l1", "item_category_l2", "item_category_l3", "item_brand",
    "item_view_count", "item_cart_count", "item_purchase_count", "item_recency",
    "category_l1_min_price", "category_l1_max_price",
    "category_l2_min_price", "category_l2_max_price",
]
item_df = train_df.drop_duplicates("product_id")[raw_item_cols].reset_index(drop=True)

# Generate embeddings via raw pyfunc (transforms + vocab built in)
embeddings_df = item_model.predict(item_df)
embeddings_np = embeddings_df.values

# Build output DataFrame
emb_output = item_df[["product_id", "item_category_l1", "item_category_l2",
                       "item_category_l3", "item_brand", "item_price"]].copy()
emb_output["embedding"] = [emb.tolist() for emb in embeddings_np]

# Write to Delta
spark_emb_df = spark.createDataFrame(emb_output)
spark_emb_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(EMBEDDING_TABLE)
spark.sql(f"ALTER TABLE {EMBEDDING_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

print(f"Saved {len(emb_output):,} item embeddings to {EMBEDDING_TABLE}")
print(f"Embedding dim: {embeddings_np.shape[1]}")
print(f"L2 norm check: {np.linalg.norm(embeddings_np[0]):.4f}")

Create or sync the Vector Search index from the embeddings Delta table.

In [0]:
vsc = VectorSearchClient()

try:
    idx_info = vsc.get_index(endpoint_name=VS_ENDPOINT, index_name=VS_INDEX)
    print(f"Index {VS_INDEX} exists. Syncing...")
    idx_info.sync()
except Exception as e:
    if any(kw in str(e).lower() for kw in ["not found", "does not exist", "resource_does_not_exist"]):
        print(f"Creating index {VS_INDEX}...")
        vsc.create_delta_sync_index(
            endpoint_name=VS_ENDPOINT,
            index_name=VS_INDEX,
            source_table_name=EMBEDDING_TABLE,
            primary_key="product_id",
            embedding_vector_column="embedding",
            embedding_dimension=EMBED_DIM,
            pipeline_type="TRIGGERED",
        )
        print(f"Index {VS_INDEX} created on endpoint {VS_ENDPOINT}")
    else:
        raise

print(f"\nVector Search index sync initiated.")
print(f"  Endpoint: {VS_ENDPOINT}")
print(f"  Index:    {VS_INDEX}")
print(f"  Source:   {EMBEDDING_TABLE}")

### Log retrieval pyfunc

Log the user tower retrieval pyfunc (embedding + VS) via `fe.log_model` with feature lineage, then register to Unity Catalog. Feature spec is extracted dynamically from the Stage 1 user tower model — no hardcoded feature lists.

In [0]:
def _load_class(fpath, cls_name):
    spec = importlib.util.spec_from_file_location(cls_name, fpath)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return getattr(mod, cls_name)
UserTowerRetrievalPyfunc = _load_class(
    os.path.join(_project_root, "user_retrieval_pyfunc.py"), "UserTowerRetrievalPyfunc"
)

# --- 1. Download Stage 1 user tower artifacts ---
user_artifact_dir = mlflow.artifacts.download_artifacts(
    run_id=STAGE1_RUN_ID, artifact_path="user_tower_pyfunc"
)
_user_raw_model = os.path.join(user_artifact_dir, "data", "feature_store", "raw_model")
user_tower_traced_path = os.path.join(_user_raw_model, "artifacts", "user_tower_traced.pt")
user_config_path = os.path.join(_user_raw_model, "artifacts", "pyfunc_config.json")

# --- 2. Extract feature spec from Stage 1 (no hardcoded names) ---
feature_spec_path = os.path.join(
    user_artifact_dir, "data", "feature_store", "feature_spec.yaml"
)
with open(feature_spec_path) as f:
    feature_spec = yaml.safe_load(f)

# Reconstruct Feature objects from the spec
feature_names = feature_spec["features"]  # fully-qualified names
features = [fe.get_feature(full_name=fn) for fn in feature_names]
print(f"Extracted {len(features)} features from Stage 1 user tower spec:")
for fn in feature_names:
    print(f"  {fn.split('.')[-1]}")

# --- 3. Reconstruct training set for fe.log_model (schema only) ---
# Parse input_columns from the spec to rebuild the labeled DataFrame
input_col_info = {}
for col_dict in feature_spec["input_columns"]:
    for col_name, col_props in col_dict.items():
        input_col_info[col_name] = col_props

exclude_cols = [k for k, v in input_col_info.items() if not v["include"]]

# Build a 1-row sample with ONLY the raw labeled columns (not feature cols).
# The session train table has FV columns already joined which would cause
# ambiguous references when fe.create_training_set joins them again.
required_cols = list(input_col_info.keys()) + ["event_type"]
labeled_sample = spark.table(SESSION_TRAIN_TABLE).limit(1)

# Add any missing excluded columns (price, session_first_date were dropped)
for col_name, props in input_col_info.items():
    if col_name not in labeled_sample.columns:
        cast_type = {"double": "double", "date": "date", "int": "int",
                     "string": "string", "timestamp": "timestamp"}.get(props["data_type"], "string")
        labeled_sample = labeled_sample.withColumn(col_name, F.lit(None).cast(cast_type))

labeled_sample = labeled_sample.select(*required_cols)

log_training_set = fe.create_training_set(
    df=labeled_sample,
    features=features,
    label="event_type",
    exclude_columns=exclude_cols,
)
print(f"Training set reconstructed (exclude: {exclude_cols})")

# --- 4. Log retrieval pyfunc + register to UC ---
mlflow.end_run()
with mlflow.start_run(run_name="two_tower_stage2_deploy") as run:
    deploy_run_id = run.info.run_id

    mlflow.log_params({
        "stage": "deployment",
        "source_stage1_run_id": STAGE1_RUN_ID,
        "vs_endpoint": VS_ENDPOINT,
        "vs_index": VS_INDEX,
        "embedding_table": EMBEDDING_TABLE,
        "endpoint_name": ENDPOINT_NAME,
        "num_features": len(features),
    })

    fe.log_model(
        model=UserTowerRetrievalPyfunc(),
        artifact_path="user_tower_retrieval",
        flavor=mlflow.pyfunc,
        training_set=log_training_set,
        artifacts={
            "user_tower": user_tower_traced_path,
            "config": user_config_path,
        },
        code_paths=[transforms_module_path, retrieval_pyfunc_module],
        pip_requirements=["torch", "numpy", "pandas", "databricks-vectorsearch"],
        resources=[DatabricksVectorSearchIndex(index_name=VS_INDEX)],
    )

    # Register to UC (adds a version to existing model, or creates if new)
    model_uri = f"runs:/{deploy_run_id}/user_tower_retrieval"
    try:
        mv = mlflow.register_model(model_uri, UC_MODEL_NAME)
        print(f"Registered {UC_MODEL_NAME} version {mv.version}")
    except Exception as e:
        print(f"Registration note: {e}")

    print(f"\nStage 2 run: {deploy_run_id}")
    print(f"Source Stage 1 run: {STAGE1_RUN_ID}")

### Validate feature lookups

Validate the registered model's feature lookups before deploying. Confirms that FE resolves entity keys → features correctly for the retrieval pyfunc logged in Stage 2.

In [0]:
# 1. Check registered model version
versions = list(w.model_versions.list(full_name=UC_MODEL_NAME))
latest_version = max(v.version for v in versions)
print(f"UC model: {UC_MODEL_NAME} v{latest_version}")
print(f"  Run ID: {versions[-1].run_id}")

# 2. Download and inspect the Stage 2 model's feature spec
stage2_dir = mlflow.artifacts.download_artifacts(
    artifact_uri=f"models:/{UC_MODEL_NAME}/{latest_version}"
)
stage2_spec_path = os.path.join(stage2_dir, "data", "feature_store", "feature_spec.yaml")
with open(stage2_spec_path) as f:
    stage2_spec = yaml.safe_load(f)

print(f"\nFeature spec ({len(stage2_spec['features'])} features):")
for fn in stage2_spec["features"]:
    print(f"  {fn.split('.')[-1]}")

include_cols = [k for col in stage2_spec["input_columns"]
                for k, v in col.items() if v["include"]]
exclude_cols = [k for col in stage2_spec["input_columns"]
                for k, v in col.items() if not v["include"]]
print(f"\nInput columns (include): {include_cols}")
print(f"Excluded columns:        {exclude_cols}")

# 3. Test feature lookup via fe.score_batch on the Stage 1 user tower
#    (embedding-only — no VS needed, same feature spec as retrieval pyfunc)
from pyspark.sql import functions as F

scoring_sample = (
    spark.table(f"{CATALOG}.{SCHEMA}.{config['ecommerce_table']}")
    .filter(F.to_date("event_time") == config["training_end_day"])
    .select("product_id", "user_session", "event_time", "event_type",
            "category_l1", "category_l2", "price")
    .withColumn("session_first_date", F.to_date("event_time"))
    .withColumn("relevance", F.lit(1))
    .withColumn("session_cart_value", F.lit(0.0))
    .withColumn("hour_of_day", F.hour("event_time"))
    .withColumn("day_of_week", F.dayofweek("event_time") - 1)
    .limit(3)
)

user_tower_uri = f"runs:/{STAGE1_RUN_ID}/user_tower_pyfunc"
print(f"\nTesting feature lookup via fe.score_batch (Stage 1 user tower)...")
user_preds = fe.score_batch(model_uri=user_tower_uri, df=scoring_sample)

_input_cols = set(scoring_sample.columns)
pred_cols = [c for c in user_preds.columns if c not in _input_cols]
print(f"  Input:  {scoring_sample.count()} rows, {len(scoring_sample.columns)} cols")
print(f"  Output: {user_preds.count()} rows, {len(user_preds.columns)} cols")
print(f"  Looked-up features ({len(pred_cols)}): {pred_cols[:6]}...")
print(f"\n\u2713 Feature lookups verified — ready to deploy.")

### Deploy endpoint

Deploy the user tower retrieval pyfunc as a serving endpoint. Feature lookup happens automatically via `fe.log_model` lineage. Route optimization enabled for low-latency inference.

In [0]:
# Get latest model version
versions = list(w.model_versions.list(full_name=UC_MODEL_NAME))
latest_version = max(v.version for v in versions)
print(f"Deploying {UC_MODEL_NAME} version {latest_version}")

# Delete existing endpoint if present
try:
    w.serving_endpoints.delete(name=ENDPOINT_NAME)
    print(f"Deleted existing endpoint: {ENDPOINT_NAME}")
    time.sleep(5)
except Exception:
    pass

# Service principal credentials for Vector Search access at serving time.
# The pyfunc reads DATABRICKS_HOST/CLIENT_ID/CLIENT_SECRET from env vars.
# Store these in a Databricks secret scope and reference them here.
SP_SECRET_SCOPE = config["sp_secret_scope"]

# Create endpoint with route optimization + SP env vars
w.serving_endpoints.create(
    name=ENDPOINT_NAME,
    config=EndpointCoreConfigInput(
        name=ENDPOINT_NAME,
        served_entities=[
            ServedEntityInput(
                entity_name=UC_MODEL_NAME,
                entity_version=latest_version,
                workload_size="Small",
                scale_to_zero_enabled=True,
                environment_vars={
                    "DATABRICKS_HOST": config["workspace_host"],
                    "DATABRICKS_CLIENT_ID": f"{{{{secrets/{SP_SECRET_SCOPE}/client-id}}}}",
                    "DATABRICKS_CLIENT_SECRET": f"{{{{secrets/{SP_SECRET_SCOPE}/client-secret}}}}",
                },
            )
        ],
    ),
    route_optimized=True,
)

print(f"\nEndpoint '{ENDPOINT_NAME}' created with route optimization.")
print(f"Waiting for deployment...")

# Poll until ready
for i in range(40):
    time.sleep(30)
    ep = w.serving_endpoints.get(name=ENDPOINT_NAME)
    state = ep.state
    print(f"  [{i*30}s] ready={state.ready}, config={state.config_update}")
    if str(state.ready) == "READY":
        print(f"\nEndpoint READY: {ENDPOINT_NAME}")
        print(f"  Model: {UC_MODEL_NAME} v{latest_version}")
        print(f"  Route optimized: True")
        break
else:
    print("Timeout waiting for endpoint. Check the Serving UI.")

### End-to-end test

Send a request to the deployed endpoint. Feature lookup happens automatically from the online store. `top_k` is configurable via params — pass it in the serving request to control how many products are retrieved from the VS index (default: 10).

In [0]:
# Grant SP access to feature tables and online stores.
# Uses the SP's application ID (resolved from mlc-scope) since UC
# may not find the SP by display name alone.
SP_SECRET_SCOPE = config["sp_secret_scope"]
client_id = dbutils.secrets.get(SP_SECRET_SCOPE, "client-id")

for grant in [
    f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `{client_id}`",
    f"GRANT USE SCHEMA ON SCHEMA {CATALOG}.{SCHEMA} TO `{client_id}`",
    f"GRANT SELECT ON SCHEMA {CATALOG}.{SCHEMA} TO `{client_id}`",
    f"GRANT EXECUTE ON SCHEMA {CATALOG}.{SCHEMA} TO `{client_id}`",
]:
    spark.sql(grant)
    print(f"OK: {grant.split('GRANT ')[1].split(' TO')[0]}")

print(f"\nSP {client_id} granted access to {CATALOG}.{SCHEMA}.")

In [0]:
import databricks.sdk.core as client

# Authenticate as SP (same credentials used for Vector Search at serving time).
# The data_plane API handles route-optimized URL routing + auth automatically.
SP_SECRET_SCOPE = config["sp_secret_scope"]

sp_config = client.Config(
    host=config["workspace_host"],
    client_id=dbutils.secrets.get(SP_SECRET_SCOPE, "client-id"),
    client_secret=dbutils.secrets.get(SP_SECRET_SCOPE, "client-secret"),
)
sp_w = WorkspaceClient(config=sp_config)

response = sp_w.serving_endpoints_data_plane.query(
    ENDPOINT_NAME,
    dataframe_records=[{
        "user_session": "test_session_001",
        "product_id": 1005115,
        "category_l1": "electronics",
        "category_l2": "smartphone",
        "relevance": 1,
        "session_cart_value": 0.0,
        "hour_of_day": 14,
        "day_of_week": 3,
    }],
)

print("Endpoint response:")
if hasattr(response, "predictions") and response.predictions:
    preds = response.predictions
    out = pd.DataFrame(preds[0] if isinstance(preds[0], list) else preds)
    print(f"Retrieved {len(out)} products")
    display(out)
else:
    print(response)